In [ ]:
pip install gensim

In [ ]:
import pandas as pd
import nltk
import gensim 
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import os


In [ ]:
def main():
    print("Loading sentiment analysis dataset...")
    df = pd.read_csv('sentiment_analysis_results.csv')
    print("Dataset loaded successfully.")

    df = pd.read_csv('sentiment_analysis_results.csv')
    df['cleaned_review'] = df['cleaned_review'].fillna('').astype(str)
    sentences = [text.split() for text in df['cleaned_review']]


    print("training word2vec model...")

    model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)

    os.makedirs('models', exist_ok=True)
    model.save('models/word2vec.model')
    print("Model trained and saved successfully.")

    pos_sentences = [text.split() for text in df[df['vader_sentiment_category'] == 'positive' ]['cleaned_review']]
    pos_model = Word2Vec(pos_sentences, vector_size=100, window=5, min_count=1, workers=4)
    neg_sentences = [text.split() for text in df[df['vader_sentiment_category'] == 'negative' ]['cleaned_review']]

    if len(neg_sentences) > 10:
        neg_model = Word2Vec(neg_sentences, vector_size=100, window=5, min_count=1, workers=4)

        target_word = 'good'
        print(f"Finding similar words to '{target_word}'")

        if target_word in pos_model.wv:
            pos_similar_words = pos_model.wv.most_similar(target_word, topn=10)
            print(f"Top 10 similar words to '{target_word}' in positive reviews:")
            for word, similarity in pos_similar_words:
                print(f"{word}: {similarity}")

        if target_word in neg_model.wv:
            neg_similar_words = neg_model.wv.most_similar(target_word, topn=10)
            print(f"Top 10 similar words to '{target_word}' in negative reviews:")
            for word, similarity in neg_similar_words:
                print(f"{word}: {similarity}")

    else:
        print("Not enough negative reviews to train a separate model.")

    target_words = ['good', 'bad', 'excellent', 'poor', 'happy', 'sad']
    results = []
    for word in target_words:
        if word in model.wv:
            similar_words = model.wv.most_similar(word, topn=10)
            results.append((word, similar_words))

    pd.DataFrame(results, columns=['Target Word', 'Similar Words']).to_csv('similar_words_results.csv', index=False)

if __name__ == "__main__":
    main()